# Training admission telemetry: 4-GPU DDP E2E

This notebook runs the bounded end-to-end check used by the PR. It trains the same deterministic model twice (telemetry disabled and enabled), then requires exact checkpoint equality and one rank-local trace per microbatch. It does not download a model or dataset.

Run the notebook from an AstrAI checkout with its normal dependencies installed. Four visible CUDA devices are required. Set `ASTRAI_E2E_GPUS` before this notebook if the physical devices are not `0,1,2,3`.

In [ ]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

repo = Path.cwd().resolve()
while repo.parent != repo and not (repo / "pyproject.toml").exists():
    repo = repo.parent
assert (repo / "scripts/eval/training_telemetry_ddp_e2e.py").is_file(), (
    "Run this notebook from the AstrAI repository."
)
print(f"repository: {repo}")
print(f"python: {sys.executable}")

## Select GPUs and verify they are visible

The test process receives only the devices listed below. It performs real DDP training and releases them when the command exits.

In [ ]:
gpu_ids = os.environ.get("ASTRAI_E2E_GPUS", "0,1,2,3")
selected_gpus = [item.strip() for item in gpu_ids.split(",") if item.strip()]
assert len(selected_gpus) == 4, f"expected four GPU IDs, got {selected_gpus}"
gpu_state = subprocess.run(
    [
        "nvidia-smi",
        "-i",
        gpu_ids,
        "--query-gpu=index,name,utilization.gpu,memory.used,memory.total",
        "--format=csv,noheader",
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(gpu_state.stdout)

## Run deterministic baseline and telemetry training

The script creates an isolated temporary output directory, streams the complete log, and exits non-zero if checkpoints differ, any rank omits a trace, token accounting is wrong, or CUDA peak-memory evidence is absent.

In [ ]:
run_parent = Path(tempfile.mkdtemp(prefix="astrai-telemetry-notebook-"))
output_dir = run_parent / "run"  # The E2E script requires a new path.
command = [
    sys.executable,
    str(repo / "scripts/eval/training_telemetry_ddp_e2e.py"),
    "--world-size",
    "4",
    "--steps",
    "4",
    "--batch-per-device",
    "2",
    "--sequence-length",
    "32",
    "--output-dir",
    str(output_dir),
]
environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = gpu_ids
environment["PYTHONPATH"] = str(repo)
environment["PYTHONUNBUFFERED"] = "1"
print("running:", " ".join(command))
process = subprocess.Popen(
    command,
    cwd=repo,
    env=environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
assert return_code == 0, f"E2E failed with exit code {return_code}"

## Inspect the independently written rank traces

In [ ]:
trace_dir = output_dir / "telemetry" / "traces"
trace_counts = {}
peak_hbm_bytes = {}
for rank in range(4):
    path = trace_dir / f"rank-{rank}.jsonl"
    records = []
    for line in path.read_text(encoding="utf-8").splitlines():
        prefix = "training_telemetry "
        if line.startswith(prefix):
            records.append(json.loads(line.removeprefix(prefix)))
    trace_counts[rank] = len(records)
    peak_hbm_bytes[rank] = max(record["peak_hbm_bytes"] for record in records)
assert trace_counts == {0: 4, 1: 4, 2: 4, 3: 4}
assert all(value > 0 for value in peak_hbm_bytes.values())
{"trace_counts": trace_counts, "peak_hbm_bytes": peak_hbm_bytes}

## Recorded reference run

A clean run on 2026-09-04 used AstrAI commit `70772a442a5c692ac2c819ee6323f447f8422f7b`, PyTorch `2.11.0+cu128`, and four NVIDIA L20 GPUs. Each rank emitted four traces with a peak of `21,373,440` allocated bytes. Telemetry-off and telemetry-on checkpoints were exactly equal (`max_parameter_abs_diff = 0.0`) with matching SHA-256 `0a57a817a744babc3896b81abec802932e24c2b85d6bb50630a105709acc1d62`.